In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd

import torch
import numpy as np
import os
import time
import json

from model import MultiModeMultiWavelengthModel
from config import Config
from mask_loader import MaskLoader
from label_utils import visualize_labels
from simulator import Simulator

start_time = time.time()

# 设置随机种子，确保结果可重现
torch.manual_seed(42)
np.random.seed(42)

print("=" * 60)
print("多模式多波长光场调制系统 - 训练-仿真集成版 (多波长支持 - 波长标记显示版)")
print("=" * 60)

# ===== 创建增强配置 =====

# 基本参数
num_modes = 3                                    # 模式数量
wavelengths = np.array([1310e-9, 1550e-9])      # 波长列表(m) - 恢复两个波长
base_wavelength_idx = 1                          # 基准波长索引

# 空间参数
field_size = 50                                  # 场大小(像素)
layer_size = 300                                 # 层大小(像素)
focus_radius = 10                                # 焦点半径(像素)
detectsize = 40                                  # 检测区域大小(像素)

# 物理参数
z_layers = 40e-6                                 # 层间距离(m)
z_prop = 150e-6                                  # 传播距离(m)
z_step = 20e-6                                   # 传播步长(m)
pixel_size = 1e-6                                # 像素大小(m)

# 检测区域偏移
offsets = [(0,0), (0,0)]                         # 每个波长的检测区域偏移

# 训练参数
learning_rate = 0.01                             # 学习率
lr_decay = 0.99                                  # 学习率衰减
epochs = 700                                     # 训练轮数
batch_size = 16                                  # 批量大小

# Zero Padding 参数
padding_ratio = 0.01                             # Padding 比例 (1%)
use_apodization = True                           # 启用边界衰减
apodization_width = 15                           # 衰减宽度

# MaskLoader 参数
fallback_focal_lengths = [40e-6, 60e-6, 80e-6, 100e-6, 120e-6]  # 备用掩码的焦距列表
default_num_layers = 3                           # 默认层数

# 保存参数
save_dir = f"./results/{len(wavelengths)}_wl_basewl_{wavelengths[base_wavelength_idx]}_z_prop_{z_prop}_focus_{focus_radius}/"
flag_savemat = True

# ===== 创建Config对象 =====
config = Config(
    num_modes=num_modes,
    wavelengths=wavelengths,
    base_wavelength_idx=base_wavelength_idx,
    field_size=field_size,
    layer_size=layer_size,
    focus_radius=focus_radius,
    detectsize=detectsize,
    z_layers=z_layers,
    z_prop=z_prop,
    z_step=z_step,
    pixel_size=pixel_size,
    offsets=offsets,
    learning_rate=learning_rate,
    lr_decay=lr_decay,
    epochs=epochs,
    batch_size=batch_size,
    padding_ratio=padding_ratio,
    use_apodization=use_apodization,
    apodization_width=apodization_width,
    fallback_focal_lengths=fallback_focal_lengths,
    default_num_layers=default_num_layers,
    save_dir=save_dir,
    flag_savemat=flag_savemat
)

print(f"✅ 配置创建成功！")
print(f"波长数量: {len(config.wavelengths)}")
print(f"模式数量: {config.num_modes}")
print(f"保存目录: {config.save_dir}")

def load_field_data_multiwavelength(config):
    """加载多波长光场数据"""
    
    print("🔍 加载多波长光场数据...")
    
    # 查找所有.npy文件
    field_files = []
    base_dir = config.save_dir
    
    for file in os.listdir(base_dir):
        if file.endswith('.npy'):
            field_files.append(os.path.join(base_dir, file))
    
    print(f"✅ 找到 {len(field_files)} 个光场数据文件")
    
    # 按波长、层数和模式组织数据: wavelength -> layer -> mode -> data_list
    field_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    
    for file_path in field_files:
        filename = os.path.basename(file_path)
        
        try:
            # 解析文件名获取波长、层数和模式信息
            parts = filename.replace('.npy', '').split('_')
            
            wavelength_nm = None
            mode_num = None
            layer_num = None
            
            for part in parts:
                if part.endswith('nm'):
                    wavelength_nm = int(part.replace('nm', ''))
                elif part.startswith('mode'):
                    mode_num = int(part.replace('mode', ''))
                elif part.endswith('layers'):
                    layer_num = int(part.replace('layers', ''))
            
            if wavelength_nm is None or mode_num is None or layer_num is None:
                print(f"❌ 跳过文件 (解析失败): {filename}")
                continue
            
            # 加载光场数据
            field = np.load(file_path)
            
            if field.size == 0:
                continue
            
            # 计算强度
            if np.iscomplexobj(field):
                intensity = np.abs(field)**2
            else:
                intensity = field**2
            
            field_data[wavelength_nm][layer_num][mode_num].append({
                'intensity': intensity,
                'field': field,
                'filename': filename,
                'wavelength': wavelength_nm,
                'layer': layer_num,
                'mode': mode_num,
                'max_intensity': np.max(intensity),
                'total_power': np.sum(intensity),
                'mean_intensity': np.mean(intensity)
            })
            
            print(f"  ✓ {wavelength_nm}nm {layer_num}层 模式{mode_num}: {field.shape}, 最大强度: {np.max(intensity):.6f}")
            
        except Exception as e:
            print(f"❌ 处理失败 {filename}: {e}")
            continue
    
    return field_data

def get_wavelength_detector_labels(wavelengths_nm, num_modes):
    """生成波长检测器标签映射"""
    # 按波长排序
    sorted_wavelengths = sorted(wavelengths_nm)
    
    # 创建标签映射：检测器索引 -> 波长标记
    detector_labels = {}
    for wl_idx, wavelength_nm in enumerate(sorted_wavelengths):
        for mode_idx in range(num_modes):
            detector_idx = wl_idx * num_modes + mode_idx
            detector_labels[detector_idx] = f'{wavelength_nm}nm-Det{mode_idx+1}'
    
    return detector_labels

def visualize_individual_mode_regions_wavelength_display(field_data, wavelength, layer_num, evaluation_regions, detector_labels, save_dir=None):
    """为每个模式单独可视化检测区域 - 波长标记显示版本（保持原有区域创建逻辑）"""
    
    print(f"   🎨 为{wavelength}nm第{layer_num}层的每个模式生成波长标记显示图片...")
    
    if wavelength not in field_data or layer_num not in field_data[wavelength]:
        print(f"   ❌ {wavelength}nm第{layer_num}层无数据")
        return
    
    layer_data = field_data[wavelength][layer_num]
    
    # 为每个模式单独生成图片
    for mode_idx in range(1, 4):  # 模式1, 2, 3
        if mode_idx not in layer_data or not layer_data[mode_idx]:
            print(f"   ⚠️ 模式{mode_idx}无数据")
            continue
        
        # 获取该模式的强度数据
        mode_data = layer_data[mode_idx][0]  # 取第一个样本
        intensity = mode_data['intensity']
        
        # 创建图片
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # 左图：该模式的纯强度分布
        im1 = ax1.imshow(intensity, cmap='hot', origin='upper')
        ax1.set_title(f"{wavelength}nm Layer {layer_num} - Mode {mode_idx} Intensity")
        ax1.axis('off')
        plt.colorbar(im1, ax=ax1, shrink=0.8)
        
        # 右图：该模式的强度 + 所有检测区域（显示波长标记）
        im2 = ax2.imshow(intensity, cmap='hot', origin='upper')
        ax2.set_title(f"{wavelength}nm Layer {layer_num} - Mode {mode_idx} + Detection Regions")
        ax2.axis('off')
        
        # 绘制所有检测区域，使用波长标记显示
        colors = ['cyan', 'lime', 'yellow', 'magenta', 'orange', 'red']
        for i, (x_start, x_end, y_start, y_end) in enumerate(evaluation_regions):
            color = colors[i % len(colors)]
            
            # 绘制矩形框
            rect = plt.Rectangle((x_start, y_start), x_end - x_start, y_end - y_start,
                               linewidth=2, edgecolor=color, facecolor='none')
            ax2.add_patch(rect)
            
            # 添加波长标记标签（仅用于显示）
            center_x = (x_start + x_end) / 2
            center_y = (y_start + y_end) / 2
            
            # 使用波长标记显示
            if i in detector_labels:
                label_text = detector_labels[i]
            else:
                label_text = f'Det{i+1}'
                
            ax2.text(center_x, center_y, label_text, 
                    ha='center', va='center', fontsize=7, 
                    color='white', weight='bold')
        
        plt.colorbar(im2, ax=ax2, shrink=0.8)
        plt.tight_layout()
        
        # 保存图片
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            filename = f'{wavelength}nm_layer_{layer_num}_mode_{mode_idx}_wavelength_display.png'
            save_path = os.path.join(save_dir, filename)
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"   ✅ 保存: {filename}")
        
        plt.close()

def visualize_all_modes_combined_wavelength_display(field_data, wavelength, layer_num, evaluation_regions, detector_labels, save_dir=None):
    """显示所有模式的合成图 - 波长标记显示版本"""
    
    print(f"   🎨 生成{wavelength}nm第{layer_num}层的波长标记显示合成图...")
    
    if wavelength not in field_data or layer_num not in field_data[wavelength]:
        return
    
    layer_data = field_data[wavelength][layer_num]
    
    # 合成所有模式的强度
    combined_intensity = None
    valid_modes = 0
    
    for mode_idx in range(1, 4):
        if mode_idx in layer_data and layer_data[mode_idx]:
            mode_intensity = layer_data[mode_idx][0]['intensity']
            
            if combined_intensity is None:
                combined_intensity = np.zeros_like(mode_intensity)
            
            combined_intensity += mode_intensity
            valid_modes += 1
    
    if combined_intensity is None:
        print(f"   ❌ {wavelength}nm第{layer_num}层无有效数据")
        return
    
    # 创建合成图
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # 左图：合成强度
    im1 = ax1.imshow(combined_intensity, cmap='hot', origin='upper')
    ax1.set_title(f"{wavelength}nm Layer {layer_num} - Combined Intensity ({valid_modes} modes)")
    ax1.axis('off')
    plt.colorbar(im1, ax=ax1, shrink=0.8)
    
    # 右图：合成强度 + 检测区域（波长标记显示）
    im2 = ax2.imshow(combined_intensity, cmap='hot', origin='upper')
    ax2.set_title(f"{wavelength}nm Layer {layer_num} - Combined + Detection Regions")
    ax2.axis('off')
    
    # 绘制检测区域（使用波长标记显示）
    colors = ['cyan', 'lime', 'yellow', 'magenta', 'orange', 'red']
    for i, (x_start, x_end, y_start, y_end) in enumerate(evaluation_regions):
        color = colors[i % len(colors)]
        
        rect = plt.Rectangle((x_start, y_start), x_end - x_start, y_end - y_start,
                           linewidth=2, edgecolor=color, facecolor='none')
        ax2.add_patch(rect)
        
        center_x = (x_start + x_end) / 2
        center_y = (y_start + y_end) / 2
        
        # 使用波长标记显示
        if i in detector_labels:
            label_text = detector_labels[i]
        else:
            label_text = f'Det{i+1}'
            
        ax2.text(center_x, center_y, label_text, 
                ha='center', va='center', fontsize=8, 
                color='white', weight='bold')
    
    plt.colorbar(im2, ax=ax2, shrink=0.8)
    plt.tight_layout()
    
    # 保存合成图
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        filename = f'{wavelength}nm_layer_{layer_num}_combined_wavelength_display.png'
        save_path = os.path.join(save_dir, filename)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   ✅ 保存合成图: {filename}")
    
    plt.close()

def evaluate_output(intensity_field, evaluation_regions):
    """
    计算每个检测区域内的平均强度
    
    参数:
        intensity_field: 2D numpy数组，强度分布
        evaluation_regions: 列表，每个元素为 (x_start, x_end, y_start, y_end)
    
    返回:
        region_intensities: 每个区域的平均强度列表
    """
    region_intensities = []
    
    for region in evaluation_regions:
        x_start, x_end, y_start, y_end = region
        
        # 确保索引在有效范围内
        x_start = max(0, min(x_start, intensity_field.shape[1]-1))
        x_end = max(0, min(x_end, intensity_field.shape[1]))
        y_start = max(0, min(y_start, intensity_field.shape[0]-1))
        y_end = max(0, min(y_end, intensity_field.shape[0]))
        
        # 提取区域并计算平均强度
        if x_end > x_start and y_end > y_start:
            region_data = intensity_field[y_start:y_end, x_start:x_end]
            avg_intensity = np.mean(region_data)
            region_intensities.append(avg_intensity)
        else:
            region_intensities.append(0.0)
    
    return np.array(region_intensities)

def calculate_intensity_cross_matrix_multiwavelength(field_data, config):
    """多波长交叉矩阵计算 - 保持原有逻辑，仅显示时使用波长标记"""
    
    print("\n📊 计算多波长强度交叉矩阵...")
    
    # 导入检测区域创建函数
    from data_generator import create_evaluation_regions_by_wavelength
    
    # 获取参数
    num_modes = config.num_modes
    focus_radius = config.focus_radius
    detectsize = config.detectsize
    num_wavelengths = len(config.wavelengths)
    wavelengths_nm = [int(wl * 1e9) for wl in config.wavelengths]
    
    # 获取所有波长、层数和模式
    all_wavelengths = sorted(field_data.keys())
    all_layers = sorted(set().union(*[wl_data.keys() for wl_data in field_data.values()]))
    all_modes = sorted(set().union(*[
        set().union(*[layer_data.keys() for layer_data in wl_data.values()])
        for wl_data in field_data.values()
    ]))
    
    # 计算总检测区域数量：波长数 × 模式数
    total_detectors = num_wavelengths * num_modes
    
    # 创建波长检测器标签映射
    detector_labels = get_wavelength_detector_labels(wavelengths_nm, num_modes)
    
    print(f"📋 分析计划: {len(all_wavelengths)}个波长 × {len(all_layers)}个层配置 × {len(all_modes)}个模式")
    print(f"波长: {all_wavelengths}nm")
    print(f"层数: {all_layers}")
    print(f"模式: {all_modes}")
    print(f"总检测区域数量: {total_detectors} ({num_wavelengths} 波长 × {num_modes} 模式)")
    print(f"检测器标记: {list(detector_labels.values())}")
    
    # 结果存储: layer -> matrix (跨波长的统一矩阵)
    cross_matrices = {}
    normalized_matrices = {}
    visibility_results = {}
    
    # 处理每个层配置
    for layer_idx, layer_num in enumerate(all_layers):
        print(f"\n🔄 [{layer_idx+1}/{len(all_layers)}] 处理 {layer_num} 层配置...")
        
        try:
            # 创建跨波长的交叉矩阵: (波长数×模式数) × 模式数
            intensity_matrix = np.zeros((total_detectors, num_modes))
            
            # 获取样本数据以确定field_size
            sample_data = None
            for wavelength in all_wavelengths:
                if wavelength in field_data and layer_num in field_data[wavelength]:
                    for mode_data_list in field_data[wavelength][layer_num].values():
                        if mode_data_list:
                            sample_data = mode_data_list[0]
                            break
                    if sample_data:
                        break
            
            if sample_data is None:
                print(f"  ❌ 无有效数据")
                continue
                
            field_size = sample_data['intensity'].shape[0]
            
            # 为每个波长创建检测区域（保持原有逻辑）
            all_evaluation_regions = []
            wavelength_detector_mapping = {}  # 记录每个波长对应的检测器索引范围
            
            for wl_idx, wavelength in enumerate(all_wavelengths):
                # 为这个波长创建检测区域（使用原有函数）
                wl_evaluation_regions = create_evaluation_regions_by_wavelength(
                    field_size, field_size, focus_radius, detectsize, 
                    offsets=config.offsets, num_modes=num_modes
                )
                
                # 记录这个波长的检测器索引范围
                start_idx = wl_idx * num_modes
                end_idx = start_idx + num_modes
                wavelength_detector_mapping[wavelength] = (start_idx, end_idx, wl_evaluation_regions)
                
                all_evaluation_regions.extend(wl_evaluation_regions)
                
                print(f"    ✓ {wavelength}nm: 检测区域 {start_idx}-{end_idx-1}")
            
            # 创建保存目录
            vis_save_dir = os.path.join(config.save_dir, "detection_visualization_wavelength_display")
            
            # 处理每个波长的数据
            processed_combinations = 0
            for wavelength in all_wavelengths:
                if wavelength not in field_data or layer_num not in field_data[wavelength]:
                    continue
                
                layer_data = field_data[wavelength][layer_num]
                start_idx, end_idx, wl_evaluation_regions = wavelength_detector_mapping[wavelength]
                
                # 生成可视化图片（使用波长标记显示）
                visualize_individual_mode_regions_wavelength_display(field_data, wavelength, layer_num, wl_evaluation_regions, detector_labels, vis_save_dir)
                visualize_all_modes_combined_wavelength_display(field_data, wavelength, layer_num, wl_evaluation_regions, detector_labels, vis_save_dir)
                
                # 计算每个输入模式的强度分布
                for input_mode_idx, input_mode in enumerate(all_modes):
                    if input_mode not in layer_data:
                        continue
                        
                    mode_data_list = layer_data[input_mode]
                    if not mode_data_list:
                        continue
                    
                    # 计算区域强度
                    total_intensities = []
                    for data_entry in mode_data_list:
                        intensity = data_entry['intensity']
                        region_intensities = evaluate_output(intensity, wl_evaluation_regions)
                        total_intensities.append(region_intensities)
                    
                    if total_intensities:
                        avg_intensities = np.mean(total_intensities, axis=0)
                        
                        # 填充交叉矩阵：该波长的检测器 × 输入模式
                        for detector_idx in range(len(avg_intensities)):
                            global_detector_idx = start_idx + detector_idx
                            if input_mode_idx < num_modes and global_detector_idx < total_detectors:
                                intensity_matrix[global_detector_idx, input_mode_idx] = avg_intensities[detector_idx]
                        
                        processed_combinations += 1
                        print(f"    ✓ {wavelength}nm 模式{input_mode}: 平均强度 {np.mean(avg_intensities):.6f}")
            
            print(f"  📊 处理完成: {processed_combinations} 个波长-模式组合")
            
            # 保存和计算结果
            cross_matrices[layer_num] = intensity_matrix.copy()
            
            # 归一化：按列归一化
            normalized_matrix = np.zeros_like(intensity_matrix)
            for col in range(intensity_matrix.shape[1]):
                col_sum = np.sum(intensity_matrix[:, col])
                if col_sum > 0:
                    normalized_matrix[:, col] = intensity_matrix[:, col] / col_sum
            
            normalized_matrices[layer_num] = normalized_matrix
            
            # 计算可见度：对角线元素的平均值（需要重新定义对角线）
            # 对于多波长情况，可见度计算需要考虑每个波长内的对角线元素
            visibility_values = []
            for wl_idx, wavelength in enumerate(all_wavelengths):
                start_idx = wl_idx * num_modes
                for mode_idx in range(num_modes):
                    detector_idx = start_idx + mode_idx
                    if detector_idx < normalized_matrix.shape[0] and mode_idx < normalized_matrix.shape[1]:
                        visibility_values.append(normalized_matrix[detector_idx, mode_idx])
            
            visibility = np.mean(visibility_values) if visibility_values else 0
            visibility_results[layer_num] = visibility
            
            print(f"  🎯 可见度: {visibility:.4f}")
            
        except Exception as e:
            print(f"  ❌ 处理失败: {e}")
            continue
    
    print(f"\n✅ 多波长交叉矩阵计算完成！处理了 {len(cross_matrices)} 个配置")
    return cross_matrices, normalized_matrices, visibility_results, all_wavelengths, all_layers, all_modes, detector_labels

def plot_intensity_cross_matrices_multiwavelength_unified(normalized_matrices, visibility_results, all_wavelengths, all_layers, all_modes, detector_labels, save_dir):
    """绘制统一的多波长强度交叉矩阵 - 波长标记显示版本"""
    
    print("\n🎨 绘制波长标记显示的多波长强度交叉矩阵...")
    
    os.makedirs(save_dir, exist_ok=True)
    
    num_layers = len(all_layers)
    num_modes = len(all_modes)
    num_wavelengths = len(all_wavelengths)
    total_detectors = num_wavelengths * num_modes
    
    if num_layers == 1:
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        axes = [ax]
    else:
        fig, axes = plt.subplots(1, num_layers, figsize=(8 * num_layers, 6), constrained_layout=True)
        if num_layers == 1:
            axes = [axes]
    
    for idx, layer_num in enumerate(all_layers):
        if layer_num not in normalized_matrices:
            continue
            
        normalized_matrix = normalized_matrices[layer_num]
        
        ax = axes[idx]
        
        # 绘制热力图
        im = ax.imshow(normalized_matrix, cmap='Oranges', interpolation='nearest', 
                      vmin=0, vmax=1, origin='upper')
        
        ax.set_xlabel('Input Mode Index')
        ax.set_ylabel('Detector Regions (Wavelength × Mode)')
        
        # 设置X轴刻度标签（输入模式）
        ax.set_xticks(np.arange(num_modes))
        ax.set_xticklabels([f'Mode{m}' for m in all_modes])
        
        # 设置Y轴刻度标签（使用波长标记）
        y_labels = [detector_labels.get(i, f'Det{i+1}') for i in range(total_detectors)]
        y_ticks = list(range(total_detectors))
        
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_labels, fontsize=8)
        
        # 添加波长分隔线
        for wl_idx in range(1, num_wavelengths):
            y_pos = wl_idx * num_modes - 0.5
            ax.axhline(y=y_pos, color='white', linewidth=2, linestyle='-')
        
        # 添加数值标注
        for i in range(normalized_matrix.shape[0]):
            for j in range(normalized_matrix.shape[1]):
                value = normalized_matrix[i, j] * 100
                ax.text(j, i, f"{value:.1f}", ha='center', va='center', 
                       color='black', fontsize=8, weight='bold')
    
    # 添加colorbar
    if num_layers > 1:
        cbar = fig.colorbar(im, ax=axes, shrink=0.6, location='right')
    else:
        cbar.set_label("Normalized Intensity (%)")
    
    plt.suptitle(f"Multi-wavelength Intensity Cross Matrix (Column-normalized, in %)\n"
                f"{num_wavelengths} Wavelengths × {num_modes} Modes = {total_detectors} Detectors", 
                fontsize=14)
    
    # 保存图像
    save_path = os.path.join(save_dir, 'intensity_cross_matrix_multiwavelength_wavelength_display.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✅ 保存: intensity_cross_matrix_multiwavelength_wavelength_display.png")
    
    # 绘制可见度对比图
    if len(visibility_results) > 1:
        plt.figure(figsize=(10, 6))
        layers = list(visibility_results.keys())
        visibilities = list(visibility_results.values())
        
        plt.plot(layers, visibilities, marker='o', linestyle='-', color='orange',
                linewidth=3, markersize=10, label=f'Multi-wavelength ({num_wavelengths} WLs)')
        
        # 添加数值标注
        for x, y in zip(layers, visibilities):
            plt.text(x, y + 0.02, f"{y:.3f}", ha='center', va='bottom', 
                    fontsize=11, weight='bold')
        
        plt.ylim(0, 1)
        plt.xlabel("Number of Layers", fontsize=12)
        plt.ylabel("Visibility", fontsize=12)
        plt.title(f"Visibility vs Number of Layers\n({num_wavelengths} Wavelengths × {num_modes} Modes)", fontsize=14)
        plt.xticks(layers)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        save_path = os.path.join(save_dir, 'visibility_comparison_multiwavelength_wavelength_display.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✅ 保存: visibility_comparison_multiwavelength_wavelength_display.png")

def save_cross_matrix_data_multiwavelength_unified(cross_matrices, normalized_matrices, visibility_results, all_wavelengths, all_layers, all_modes, detector_labels, save_dir):
    """保存统一多波长交叉矩阵数据 - 波长标记显示版本"""
    
    print("\n💾 保存波长标记显示多波长交叉矩阵数据...")
    
    num_modes = len(all_modes)
    num_wavelengths = len(all_wavelengths)
    
    # 保存为numpy文件
    for layer_num in all_layers:
        if layer_num in cross_matrices:
            # 原始强度矩阵
            np.save(os.path.join(save_dir, f'intensity_matrix_multiWL_wavelength_display_{layer_num}layers_raw.npy'), 
                   cross_matrices[layer_num])
            
            # 归一化矩阵
            np.save(os.path.join(save_dir, f'intensity_matrix_multiWL_wavelength_display_{layer_num}layers_normalized.npy'), 
                   normalized_matrices[layer_num])
    
    # 保存为CSV文件
    for layer_num in all_layers:
        if layer_num in normalized_matrices:
            # 创建行标签（使用波长标记）
            total_detectors = num_wavelengths * num_modes
            row_labels = [detector_labels.get(i, f'Det{i+1}') for i in range(total_detectors)]
            
            # 创建列标签（输入模式）
            col_labels = [f'Mode{m}' for m in all_modes]
            
            df = pd.DataFrame(
                normalized_matrices[layer_num],
                index=row_labels,
                columns=col_labels
            )
            
            csv_path = os.path.join(save_dir, f'intensity_cross_matrix_multiWL_wavelength_display_{layer_num}layers.csv')
            df.to_csv(csv_path, encoding='utf-8-sig')
            
            print(f"  ✅ 保存: intensity_cross_matrix_multiWL_wavelength_display_{layer_num}layers.csv")
    
    # 保存可见度结果
    visibility_data = []
    for layer, vis in visibility_results.items():
        visibility_data.append({
            'Layers': layer,
            'Wavelengths': f"{sorted([int(wl*1e9) for wl in [1310e-9, 1550e-9]])}nm",
            'Detector_Labels': ', '.join(list(detector_labels.values())),
            'Total_Detectors': num_wavelengths * num_modes,
            'Visibility': vis
        })
    
    if visibility_data:
        visibility_df = pd.DataFrame(visibility_data)
        visibility_path = os.path.join(save_dir, 'visibility_results_multiwavelength_wavelength_display.csv')
        visibility_df.to_csv(visibility_path, index=False, encoding='utf-8-sig')
        
        print(f"  ✅ 保存: visibility_results_multiwavelength_wavelength_display.csv")

def print_cross_matrix_summary_multiwavelength_unified(cross_matrices, normalized_matrices, visibility_results, all_wavelengths, all_modes, detector_labels):
    """打印统一多波长交叉矩阵摘要 - 波长标记显示版本"""
    
    print("\n📋 波长标记显示多波长强度交叉矩阵摘要")
    print("="*60)
    
    num_modes = len(all_modes)
    num_wavelengths = len(all_wavelengths)
    total_detectors = num_wavelengths * num_modes
    
    print(f"配置信息:")
    print(f"  波长: {sorted([int(wl*1e9) for wl in [1310e-9, 1550e-9]])}nm")
    print(f"  模式数: {num_modes}")
    print(f"  检测器标记: {', '.join(list(detector_labels.values()))}")
    print(f"  总检测器数: {total_detectors} ({num_wavelengths} 波长 × {num_modes} 模式)")
    print(f"  分析配置数量: {len(cross_matrices)}")
    
    if cross_matrices:
        print(f"  层数范围: {min(cross_matrices.keys())} - {max(cross_matrices.keys())}")
    
    print(f"\n各配置可见度:")
    print("-" * 30)
    
    for layer_num in sorted(visibility_results.keys()):
        visibility = visibility_results[layer_num]
        print(f"{layer_num:2d}层: {visibility:.4f}")
    
    # 找到最佳配置
    if visibility_results:
        best_layer = max(visibility_results.keys(), key=lambda k: visibility_results[k])
        best_visibility = visibility_results[best_layer]
        
        print(f"\n🏆 最佳配置: {best_layer}层")
        print(f"   最高可见度: {best_visibility:.4f}")
        
        # 显示最佳配置的矩阵摘要
        if best_layer in normalized_matrices:
            print(f"\n最佳配置交叉矩阵摘要 ({best_layer}层):")
            print("-" * 50)
            best_matrix = normalized_matrices[best_layer]
            
            print(f"矩阵尺寸: {best_matrix.shape[0]} × {best_matrix.shape[1]}")
            print(f"检测器分布:")
            
            for wl_idx, wavelength in enumerate(sorted(all_wavelengths)):
                start_idx = wl_idx * num_modes
                end_idx = start_idx + num_modes
                wl_nm = int([wl for wl in [1310e-9, 1550e-9] if int(wl*1e9) == wavelength][0] * 1e9)
                print(f"  {wl_nm}nm: 检测器 {start_idx}-{end_idx-1} (标记: {wl_nm}nm-Det1~3)")
                
                # 显示该波长的对角线元素
                diag_values = []
                for mode_idx in range(num_modes):
                    detector_idx = start_idx + mode_idx
                    if detector_idx < best_matrix.shape[0] and mode_idx < best_matrix.shape[1]:
                        diag_values.append(best_matrix[detector_idx, mode_idx] * 100)
                
                if diag_values:
                    print(f"    对角线强度: {[f'{v:.1f}%' for v in diag_values]}")

# ===== 修改主函数 =====
def main_intensity_cross_matrix_analysis_multiwavelength(config):
    """主要的多波长强度交叉矩阵分析函数 - 波长标记显示版本"""
    
    print("\n" + "="*60)
    print("多波长光场强度交叉矩阵分析 - 波长标记显示版本")
    print("="*60)
    
    # 从 config 获取参数
    print(f"📋 使用配置参数:")
    print(f"   波长: {[int(wl*1e9) for wl in config.wavelengths]}nm")
    print(f"   模式数量: {config.num_modes}")
    print(f"   总检测器数: {len(config.wavelengths) * config.num_modes}")
    print(f"   焦点半径: {config.focus_radius}")
    print(f"   检测区域大小: {config.detectsize}")
    print(f"   注意: 保持原有检测区域创建逻辑，仅显示时使用波长标记")
    
    # 1. 加载数据
    field_data = load_field_data_multiwavelength(config)
    
    if not field_data:
        print("❌ 未找到有效的光场数据")
        return None, None, None
    
    # 2. 计算交叉矩阵
    cross_matrices, normalized_matrices, visibility_results, all_wavelengths, all_layers, all_modes, detector_labels = \
        calculate_intensity_cross_matrix_multiwavelength(field_data, config)
    
    if not cross_matrices:
        print("❌ 交叉矩阵计算失败")
        return None, None, None
    
    # 3. 创建保存目录
    save_dir = os.path.join(config.save_dir, "intensity_cross_matrix_analysis_multiwavelength_wavelength_display")
    os.makedirs(save_dir, exist_ok=True)
    
    # 4. 绘制矩阵
    plot_intensity_cross_matrices_multiwavelength_unified(
        normalized_matrices, visibility_results, all_wavelengths, all_layers, all_modes, detector_labels, save_dir)
    
    # 5. 保存数据
    save_cross_matrix_data_multiwavelength_unified(
        cross_matrices, normalized_matrices, visibility_results, all_wavelengths, all_layers, all_modes, detector_labels, save_dir)
    
    # 6. 打印摘要
    print_cross_matrix_summary_multiwavelength_unified(
        cross_matrices, normalized_matrices, visibility_results, all_wavelengths, all_modes, detector_labels)
    
    print(f"\n🎉 波长标记显示版多波长强度交叉矩阵分析完成！")
    print(f"📁 结果保存在: {save_dir}")
    print(f"📊 检测器标记: {', '.join(list(detector_labels.values()))}")
    print(f"📊 查看交叉矩阵: intensity_cross_matrix_multiwavelength_wavelength_display.png")
    print(f"📈 查看可见度对比: visibility_comparison_multiwavelength_wavelength_display.png")
    print(f"📋 查看数据表: intensity_cross_matrix_multiWL_wavelength_display_*layers.csv")
    print(f"🎨 查看检测区域可视化: detection_visualization_wavelength_display/")
    
    return cross_matrices, normalized_matrices, visibility_results

# ===== 在您的主程序最后替换原来的分析调用 =====
print("\n" + "="*60)
print("执行波长标记显示版多波长强度交叉矩阵分析")
print("="*60)

# 运行波长标记显示版本的分析
try:
    cross_matrices, normalized_matrices, visibility_results = main_intensity_cross_matrix_analysis_multiwavelength(config)
    
    if cross_matrices is not None:
        print("✅ 波长标记显示版多波长强度交叉矩阵分析成功完成！")
        
        # 显示简要结果
        print("\n📊 可见度摘要:")
        for layer_num, visibility in visibility_results.items():
            print(f"   {layer_num}层: {visibility:.4f}")
        
        print("\n🏷️ 检测器标记说明:")
        print("   原有检测区域创建逻辑保持不变")
        print("   仅在显示和保存时使用波长标记:")
        print("   1310nm-Det1, 1310nm-Det2, 1310nm-Det3")
        print("   1550nm-Det1, 1550nm-Det2, 1550nm-Det3")
        
        # 显示生成的文件总结
        print("\n📁 生成的文件:")
        print("   📊 波长标记显示交叉矩阵分析:")
        print("      - intensity_cross_matrix_multiwavelength_wavelength_display.png")
        print("      - visibility_comparison_multiwavelength_wavelength_display.png")
        print("      - intensity_cross_matrix_multiWL_wavelength_display_*layers.csv")
        print("      - visibility_results_multiwavelength_wavelength_display.csv")
        
        print("   🎨 波长标记显示检测区域可视化:")
        vis_dir = os.path.join(config.save_dir, "detection_visualization_wavelength_display")
        if os.path.exists(vis_dir):
            png_files = [f for f in os.listdir(vis_dir) if f.endswith('.png')]
            for png_file in sorted(png_files)[:10]:  # 只显示前10个文件
                print(f"      - {png_file}")
            if len(png_files) > 10:
                print(f"      - ... 还有 {len(png_files)-10} 个文件")
        
    else:
        print("❌ 分析失败，请检查数据文件")
        
except Exception as e:
    print(f"❌ 分析过程中出现错误: {e}")
    import traceback
    traceback.print_exc()

print(f"\n总运行时间: {time.time() - start_time:.2f} 秒")
print("🎉 波长标记显示版分析完成！")

多模式多波长光场调制系统 - 训练-仿真集成版 (多波长支持 - 波长标记显示版)
✓ 基准波长设置: 索引 1 -> 1550.0nm
✓ Zero Padding 配置: 填充比例=0.01, 衰减=True
  - 衰减类型: cosine, 宽度: 15像素
配置完成，使用设备: cuda
✅ 配置创建成功！
波长数量: 2
模式数量: 3
保存目录: ./results/2_wl_basewl_1.55e-06_z_prop_0.00015_focus_10/

执行波长标记显示版多波长强度交叉矩阵分析

多波长光场强度交叉矩阵分析 - 波长标记显示版本
📋 使用配置参数:
   波长: [1310, 1550]nm
   模式数量: 3
   总检测器数: 6
   焦点半径: 10
   检测区域大小: 40
   注意: 保持原有检测区域创建逻辑，仅显示时使用波长标记
🔍 加载多波长光场数据...
✅ 找到 60 个光场数据文件
  ✓ 1310nm 5层 模式2: (300, 300), 最大强度: 0.358744
  ✓ 1550nm 2层 模式2: (300, 300), 最大强度: 0.077428
  ✓ 1310nm 1层 模式3: (300, 300), 最大强度: 0.161689
  ✓ 1550nm 2层 模式1: (300, 300), 最大强度: 0.033610
  ✓ 1310nm 4层 模式2: (300, 300), 最大强度: 0.633739
  ✓ 1310nm 3层 模式2: (300, 300), 最大强度: 0.437901
  ✓ 1550nm 2层 模式3: (300, 300), 最大强度: 0.071251
  ✓ 1550nm 1层 模式2: (300, 300), 最大强度: 0.127585
  ✓ 1550nm 5层 模式3: (300, 300), 最大强度: 0.046012
  ✓ 1310nm 2层 模式1: (300, 300), 最大强度: 0.072214
  ✓ 1550nm 5层 模式2: (300, 300), 最大强度: 0.103775
  ✓ 1550nm 4层 模式2: (300, 300), 最大强度: 0.091072
  ✓ 1310nm 1层 模式2: 

Traceback (most recent call last):
  File "/tmp/ipykernel_21063/1054851553.py", line 773, in <module>
    cross_matrices, normalized_matrices, visibility_results = main_intensity_cross_matrix_analysis_multiwavelength(config)
                                                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_21063/1054851553.py", line 733, in main_intensity_cross_matrix_analysis_multiwavelength
    cross_matrices, normalized_matrices, visibility_results, all_wavelengths, all_layers, all_modes, detector_labels = \
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: not enough values to unpack (expected 7, got 2)
